# Notebook 05 (Lucas): MLP Training -- Exp 2 and Exp 5

**Exp 2:** Delta Residue only -- `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Exp 5:** Delta Residue + PCA reduction -- `PCA(delta_residue[mut_pos])`

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [1]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project


Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [2]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

Drive root:      /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir:   /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Checkpoint dir:  /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/checkpoints
Paths set.


`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [3]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

Local run -- installation skipped.


In [4]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

Autoreload enabled.


In [5]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device: mps
Apple MPS -- Apple Silicon unified memory


Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [6]:
import numpy as np
import pandas as pd
import wandb
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

Imports OK.


## Data Loading and Splits

In [7]:
df = load_abagym_antibody(DATA_DIR)

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}


Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Oscar uses the same value in NB05_oscar.ipynb
to guarantee identical splits across both notebooks.

Confirmed output:

```
Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val:   {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test:  {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
```

All 5 antibodies present in every split. HER2 val/test N=18 -- Spearman on 18 samples
is noisy; report but note unreliability. Splits are identical to Oscar's notebook.

## Experiment 2: Delta Residue Only

**Input:** `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Dims:** ESM-2 = 1280, AbLang2 = 480  
**Owner:** Lucas

The most local embedding strategy: a single token-level delta at the mutation site
only. This encodes how much the mutated amino acid shifts the contextual embedding
at that specific position. From NB04 EDA, the L2 norm of this vector is weakly
predictive. The MLP has access to the
full directional vector, not just the norm, so supervised performance should
substantially exceed the EDA baseline.

In [8]:
# Build datasets for both models -- Exp 2 (DELTA_RESIDUE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_RESIDUE: input_dim=1280, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE: input_dim=480, label=0.7380, region=CDR_H3


Sanity check: verifies input dimensions for Exp 2 (ESM-2=1280, AbLang2=480)
and that the dataset loads correctly.

Confirmed output:

```
esm2 DELTA_RESIDUE: input_dim=1280, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE: input_dim=480, label=0.7380, region=CDR_H3
```

Input dims correct. Same row 0 as Oscar's notebook (Ang2_2017_G6 H:P100A,
CDR_H3, MinMax score=0.7380).

## Experiment 5: Delta Residue + PCA Reduction

**Input:** `PCA(delta_residue[mut_pos])` reduced to `n_components` dimensions  
**Dims:** `n_components` (to be determined -- sweep suggested values below)  
**Owner:** Lucas

Same local delta residue as Exp 2, but dimensionality-reduced via PCA before
the MLP. Motivation: the raw 1280/480-dim vector may have many uninformative
dimensions that add noise. PCA retains the principal axes of variation in the
training set's delta residue space.

**Critical implementation note:** PCA must be fit on the training split only,
then applied to val and test using the same fitted transform. Fitting on the
full dataset would leak test information into the dimensionality reduction.
The `transform` parameter in `AbAgymDataset` handles this: fit PCA on train,
pass `transform=lambda x: pca.transform(x[None])[0]` to all three splits.

Suggested `n_components` to evaluate: 32, 64, 128, 256.

In [9]:
# Example: fit PCA on training split for ESM-2 delta residue
# Repeat for AbLang2 and for each n_components value

N_COMPONENTS = 64  # adjust as needed
model_name = 'esm2'

# Load raw delta residue for the full dataset to extract train vectors
ds_full = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE,
    model_name=model_name,
)

# Extract training vectors (no transform applied yet)
train_vecs = np.stack([ds_full[i][0].numpy() for i in train_idx])
print(f"Train vectors shape: {train_vecs.shape}")

# Fit PCA on train only
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(train_vecs)
print(f"PCA explained variance (first {N_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.3f}")

# Wrap as transform callable
pca_transform = lambda x: pca.transform(x[None])[0].astype('float32')

# Build dataset with transform applied
ds_reduced = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE_REDUCED,
    model_name=model_name,
    transform=pca_transform,
)
x0, y0, _ = ds_reduced[train_idx[0]]
print(f"Reduced input dim: {x0.shape[0]}")

Train vectors shape: (4256, 1280)
PCA explained variance (first 64 components): 0.810
Reduced input dim: 64


Fits PCA on the training split only, then wraps it as a transform for the dataset.
The same `pca_transform` is passed to train, val, and test dataset instances to
ensure the same projection is applied consistently.

Confirmed output (ESM-2, N_COMPONENTS=64):

```
Train vectors shape: (4256, 1280)
PCA explained variance (first 64 components): 0.810
Reduced input dim: 64
```

64 components retain 81% of variance from the 1280-dim ESM-2 residue delta space.
This is a 20x compression with good information retention. The same PCA is applied
to AbLang2 (480-dim input) in the training cell -- explained variance will differ.

## Experiment 2: Training

Run for both ESM-2 and AbLang2.

**Expected input dims:** ESM-2 = 1280, AbLang2 = 480  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)

Exp 2 is the most local strategy: a single token-level delta at the mutation
site. The MLP must learn which dimensions of the embedding shift correlate with
functional effect, without any global antibody context.

In [ ]:
exp2_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_RESIDUE -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_residue',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp2_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

Record confirmed best epoch, val Spearman, and per-dataset breakdown here.
Key comparison point: does Exp 2 (local delta only) approach the performance
of Oscar's Exp 4 (global delta sequence)? The EDA found ESM-2 assigns near-zero
delta norms at CDR H3, so expect ESM-2 Exp 2 to underperform AbLang2 Exp 2
specifically on HER2 (all CDR H3 mutations).

## Experiment 2: Test Evaluation

In [ ]:
print("=== Experiment 2: DELTA_RESIDUE -- Test Results ===")
for model_name, result in exp2_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):     {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                  {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

Record test results here. Note the ESM-2 vs AbLang2 gap for HER2 specifically.
If ESM-2 CDR H3 insensitivity is the driver, we expect AbLang2 to substantially
outperform ESM-2 for HER2 Spearman while performing comparably on the other four.

## Experiment 5: Training

PCA is fitted inside this cell for each model independently -- the sanity-check cell above is for ESM-2 only and is not used here. **The same fitted PCA is passed to train, val, and test Subset instances within each model's loop iteration**, ensuring no leakage.

Run for both ESM-2 and AbLang2, and optionally sweep over `N_COMPONENTS` (suggested: 32, 64, 128, 256). Each combination is a separate W&B run.

For a quick first run, use `N_COMPONENTS = 64` for both models.

In [ ]:
exp5_results = {}

for model_name in ('esm2', 'ablang2'):
    # --- Fit PCA on training split only ---
    ds_raw = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )
    train_vecs = np.stack([ds_raw[i][0].numpy() for i in train_idx])
    print(f"{model_name}: fitting PCA on {train_vecs.shape} train vectors")

    pca = PCA(n_components=N_COMPONENTS, random_state=42)
    pca.fit(train_vecs)
    explained = pca.explained_variance_ratio_.sum()
    print(f"  Explained variance ({N_COMPONENTS} components): {explained:.3f}")

    pca_transform = lambda x, _pca=pca: _pca.transform(x[None])[0].astype('float32')

    # --- Build datasets with transform ---
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_REDUCED,
        model_name=model_name,
        transform=pca_transform,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[train_idx[0]]
    input_dim = x0.shape[0]
    print(f"  Reduced input_dim: {input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy=f'delta_residue_pca{N_COMPONENTS}',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
        wandb_run_name=f"{model_name}_delta_residue_pca{N_COMPONENTS}",
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    result['pca'] = pca
    result['n_components'] = N_COMPONENTS
    result['explained_variance'] = float(explained)
    exp5_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print()

Record confirmed explained variance, best epoch, and val Spearman for each model.

Key question: does PCA reduction (Exp 5) improve over raw delta residue (Exp 2)?
A higher Spearman would indicate that PCA filters noise from uninformative dimensions.
If performance is similar or worse, the raw delta residue space is already low-noise.

## Experiment 5: Test Evaluation

In [ ]:
print(f"=== Experiment 5: DELTA_RESIDUE_REDUCED (PCA {N_COMPONENTS}) -- Test Results ===")
for model_name, result in exp5_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()} (explained var: {result['explained_variance']:.3f})")
    print(f"  Aggregate Spearman (all 5):     {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                  {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

Record test results here. Compare to Exp 2 test numbers to assess whether
PCA reduction helps. If time permits, re-run the training cell with different
`N_COMPONENTS` values (32, 128, 256) and record how the sweep affects both
explained variance and test Spearman.

## Summary: Exp 2 vs Exp 5

In [ ]:
rows = []
for exp_name, results_dict in [('Exp2_DeltaRes', exp2_results), (f'Exp5_PCA{N_COMPONENTS}', exp5_results)]:
    for model_name, result in results_dict.items():
        metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
        row = {
            'Experiment': exp_name,
            'Model': model_name,
            'Spearman_all': round(metrics['aggregate'], 4),
            'Spearman_excl_HER2': round(metrics['exclude_her2'], 4),
            'HER2': round(metrics['HER2'], 4),
        }
        for ds, r in metrics['per_dataset'].items():
            row[ds] = round(r, 4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

Record the summary table here. This feeds into the cross-experiment comparison.
If running multiple `N_COMPONENTS` sweeps, add a row for each.